# Off-Ball Run Value Analysis Using SkillCorner Tracking Data

This notebook explores the SkillCorner Open Tracking Dataset to analyse the value of off-ball runs in football.

The goal is to understand how different types of off-ball movements contribute to attacking value across different phases of play and defensive structures.

Dataset:
- SkillCorner Open Data (A-League matches)
- Broadcast tracking data
- Game Intelligence dynamic events

Author: Ioannis Kastritis



## 1. Load SkillCorner Open Data

In [1]:
!git clone https://github.com/SkillCorner/opendata.git

Cloning into 'opendata'...
remote: Enumerating objects: 454, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 454 (delta 92), reused 216 (delta 68), pack-reused 197 (from 1)
Receiving objects: 100% (454/454), 159.32 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (165/165), done.
Filtering content: 100% (10/10), 873.32 MiB | 13.30 MiB/s, done.


## 2. Inspect Repository Structure

In [2]:
!ls opendata

assets	docs	 mkdocs.yml  README.md	       src
data	LICENSE  notebooks   requirements.txt


In [3]:
!ls opendata/data

aggregates  matches  matches.json


## 3. Load Match Metadata

In [4]:
import json
import pandas as pd

with open("opendata/data/matches.json", "r") as f:
    matches = json.load(f)

print("Number of matches:", len(matches))
print("Type:", type(matches))
print("Keys in first match:", matches[0].keys())

Number of matches: 10
Type: <class 'list'>
Keys in first match: dict_keys(['id', 'date_time', 'home_team', 'away_team', 'status', 'competition_id', 'season_id', 'competition_edition_id'])


## 4. Convert Match Metadata to DataFrame

In [5]:
import pandas as pd

df_matches = pd.DataFrame(matches)

df_matches[["id","home_team","away_team","date_time"]]

,id,home_team,away_team,date_time
0,2017461,"{'id': 868, 'short_name': 'Melbourne V FC'}","{'id': 4177, 'short_name': 'Auckland FC'}",2025-05-17T09:35:00Z
1,2015213,"{'id': 1803, 'short_name': 'Western United'}","{'id': 4177, 'short_name': 'Auckland FC'}",2025-05-03T08:00:00Z
2,2013725,"{'id': 1803, 'short_name': 'Western United'}","{'id': 869, 'short_name': 'Sydney FC'}",2025-04-27T07:00:00Z
3,2011166,"{'id': 867, 'short_name': 'Wellington P FC'}","{'id': 868, 'short_name': 'Melbourne V FC'}",2025-04-12T05:00:00Z
4,2006229,"{'id': 2380, 'short_name': 'Melbourne City'}","{'id': 1804, 'short_name': 'Macarthur FC'}",2025-03-07T08:35:00Z
5,1996435,"{'id': 869, 'short_name': 'Sydney FC'}","{'id': 866, 'short_name': 'Adelaide United'}",2025-02-01T06:00:00Z
6,1953632,"{'id': 870, 'short_name': 'CC Mariners'}","{'id': 2380, 'short_name': 'Melbourne City'}",2024-12-31T08:00:00Z
7,1925299,"{'id': 1802, 'short_name': 'Brisbane FC'}","{'id': 871, 'short_name': 'Perth Glory'}",2024-12-21T06:00:00Z
8,1899585,"{'id': 4177, 'short_name': 'Auckland FC'}","{'id': 867, 'short_name': 'Wellington P FC'}",2024-12-07T04:00:00Z
9,1886347,"{'id': 4177, 'short_name': 'Auckland FC'}","{'id': 1805, 'short_name': 'Newcastle'}",2024-11-30T04:00:00Z


## 5. Select Example Match

In [6]:
match_id = df_matches.loc[0, "id"]
print(match_id)

2017461


## 6. Inspect Match Files

In [7]:
import os

match_folder = f"opendata/data/matches/{match_id}"
print("Match folder:", match_folder)

os.listdir(match_folder)

Match folder: opendata/data/matches/2017461


['2017461_match.json',
 '2017461_tracking_extrapolated.jsonl',
 '2017461_phases_of_play.csv',
 '2017461_dynamic_events.csv']

## 7. Load Match Metadata File

In [8]:
import json

with open(f"{match_folder}/{match_id}_match.json", "r") as f:
    match_meta = json.load(f)

print(type(match_meta))
print(match_meta.keys())

<class 'dict'>
dict_keys(['id', 'home_team_score', 'away_team_score', 'date_time', 'stadium', 'home_team', 'home_team_kit', 'away_team', 'away_team_kit', 'home_team_coach', 'away_team_coach', 'home_team_playing_time', 'away_team_playing_time', 'competition_edition', 'match_periods', 'competition_round', 'referees', 'players', 'status', 'home_team_side', 'ball', 'pitch_length', 'pitch_width'])


In [9]:
print("Match ID:", match_meta["id"])
print("Date:", match_meta["date_time"])
print("Home team:", match_meta["home_team"])
print("Away team:", match_meta["away_team"])
print("Pitch size:", match_meta["pitch_length"], "x", match_meta["pitch_width"])
print("Number of players listed:", len(match_meta["players"]))
print("Match periods:", match_meta["match_periods"])

Match ID: 2017461
Date: 2025-05-17T09:35:00Z
Home team: {'id': 868, 'name': 'Melbourne Victory Football Club', 'short_name': 'Melbourne V FC', 'acronym': 'MEL'}
Away team: {'id': 4177, 'name': 'Auckland FC', 'short_name': 'Auckland FC', 'acronym': 'AUC'}
Pitch size: 105 x 68
Number of players listed: 36
Match periods: [{'period': 1, 'name': 'period_1', 'start_frame': 2510, 'end_frame': 30210, 'duration_frames': 27700, 'duration_minutes': 46.17}, {'period': 2, 'name': 'period_2', 'start_frame': 39670, 'end_frame': 70420, 'duration_frames': 30750, 'duration_minutes': 51.25}]


In [10]:
match_meta["players"][0]

{'player_role': {'id': 13,
  'position_group': 'Wide Attacker',
  'name': 'Right Winger',
  'acronym': 'RW'},
 'start_time': '01:23:12',
 'end_time': None,
 'number': 23,
 'yellow_card': 0,
 'red_card': 0,
 'injured': False,
 'goal': 0,
 'own_goal': 0,
 'playing_time': {'total': {'minutes_tip': 4.3,
   'minutes_otip': 2.41,
   'start_frame': 62590,
   'end_frame': 70419,
   'minutes_played': 13.05,
   'minutes_played_regular_time': 13.05},
  'by_period': [{'name': 'period_2',
    'minutes_tip': 4.3,
    'minutes_otip': 2.41,
    'start_frame': 62590,
    'end_frame': 70419,
    'minutes_played': 13.05}]},
 'team_player_id': 1564070,
 'team_id': 868,
 'id': 795521,
 'first_name': 'Alexander',
 'last_name': 'Badolato',
 'short_name': 'A. Badolato',
 'birthday': '2005-02-23',
 'trackable_object': 797084,
 'gender': 'male'}

In [11]:
import pandas as pd

phases = pd.read_csv(f"{match_folder}/{match_id}_phases_of_play.csv")

print(phases.shape)
print(phases.columns.tolist())
phases.head()

(437, 44)
['index', 'match_id', 'frame_start', 'frame_end', 'time_start', 'time_end', 'minute_start', 'second_start', 'duration', 'period', 'attacking_side_id', 'team_in_possession_id', 'attacking_side', 'team_in_possession_shortname', 'n_player_possessions_in_phase', 'team_possession_loss_in_phase', 'team_possession_lead_to_goal', 'team_possession_lead_to_shot', 'team_in_possession_phase_type', 'team_in_possession_phase_type_id', 'team_out_of_possession_phase_type', 'team_out_of_possession_phase_type_id', 'x_start', 'y_start', 'channel_id_start', 'channel_start', 'third_id_start', 'third_start', 'penalty_area_start', 'x_end', 'y_end', 'channel_id_end', 'channel_end', 'third_id_end', 'third_end', 'penalty_area_end', 'team_in_possession_width_start', 'team_in_possession_width_end', 'team_in_possession_length_start', 'team_in_possession_length_end', 'team_out_of_possession_width_start', 'team_out_of_possession_width_end', 'team_out_of_possession_length_start', 'team_out_of_possession_len

,index,match_id,frame_start,frame_end,time_start,time_end,minute_start,second_start,duration,period,...,third_end,penalty_area_end,team_in_possession_width_start,team_in_possession_width_end,team_in_possession_length_start,team_in_possession_length_end,team_out_of_possession_width_start,team_out_of_possession_width_end,team_out_of_possession_length_start,team_out_of_possession_length_end
0,0,2017461,2512,2540,00:00.2,00:03.0,0,0,2.8,1,...,defensive_third,False,40.94,42.11,43.49,53.80,34.80,31.53,41.92,54.89
1,1,2017461,2540,2573,00:03.0,00:06.3,0,3,3.3,1,...,attacking_third,False,42.04,35.13,54.19,64.79,31.55,27.82,55.14,57.02
2,2,2017461,2573,2591,00:06.3,00:08.1,0,6,1.8,1,...,middle_third,False,27.69,23.51,56.97,54.32,34.82,32.71,64.84,63.16
3,3,2017461,2591,2672,00:08.1,00:16.2,0,8,8.1,1,...,attacking_third,True,32.68,29.56,63.32,80.36,23.56,22.04,54.03,26.11
4,4,2017461,2845,2901,00:33.5,00:39.1,0,33,5.6,1,...,middle_third,False,NaN,36.53,NaN,44.17,NaN,41.80,NaN,66.08


In [12]:
phases["team_in_possession_phase_type"].value_counts()

,count
team_in_possession_phase_type,
chaotic,143
create,115
finish,64
direct,44
build_up,30
set_play,21
transition,17
quick_break,3


In [13]:
phases["team_out_of_possession_phase_type"].value_counts()

,count
team_out_of_possession_phase_type,
chaotic,143
medium_block,115
low_block,64
defending_direct,44
high_block,30
defending_set_play,21
defending_transition,17
defending_quick_break,3


## 8. Inspect Tracking Data Structure

In [14]:
import json

tracking_file = f"{match_folder}/{match_id}_tracking_extrapolated.jsonl"

with open(tracking_file) as f:
    first_frame = json.loads(next(f))

print(first_frame.keys())
print(first_frame)

dict_keys(['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data'])
{'frame': 0, 'timestamp': None, 'period': None, 'ball_data': {'x': None, 'y': None, 'z': None, 'is_detected': None}, 'possession': {'player_id': None, 'group': None}, 'image_corners_projection': {'x_top_left': None, 'y_top_left': None, 'x_bottom_left': None, 'y_bottom_left': None, 'x_bottom_right': None, 'y_bottom_right': None, 'x_top_right': None, 'y_top_right': None}, 'player_data': []}


In [15]:
import json

tracking_file = f"{match_folder}/{match_id}_tracking_extrapolated.jsonl"

first_valid_frame = None

with open(tracking_file) as f:
    for line in f:
        frame_data = json.loads(line)
        if frame_data["player_data"] and frame_data["period"] is not None:
            first_valid_frame = frame_data
            break

print(first_valid_frame.keys())
print("Frame:", first_valid_frame["frame"])
print("Timestamp:", first_valid_frame["timestamp"])
print("Period:", first_valid_frame["period"])
print("Possession:", first_valid_frame["possession"])
print("Ball data:", first_valid_frame["ball_data"])
print("Number of players:", len(first_valid_frame["player_data"]))
print("Example player:", first_valid_frame["player_data"][0])

dict_keys(['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data'])
Frame: 2510
Timestamp: 00:00:00.00
Period: 1
Possession: {'player_id': None, 'group': None}
Ball data: {'x': -0.46, 'y': -0.12, 'z': 0.14, 'is_detected': True}
Number of players: 22
Example player: {'x': -38.16, 'y': 1.51, 'player_id': 51678, 'is_detected': False}


## 9. Load Dynamic Events Dataset

In [16]:
events = pd.read_csv(f"{match_folder}/{match_id}_dynamic_events.csv")

print(events.shape)
print(events.columns.tolist())

events.head()

(4188, 294)
['event_id', 'index', 'match_id', 'frame_start', 'frame_end', 'frame_physical_start', 'time_start', 'time_end', 'minute_start', 'second_start', 'duration', 'period', 'attacking_side_id', 'attacking_side', 'event_type_id', 'event_type', 'event_subtype_id', 'event_subtype', 'player_id', 'player_name', 'player_position_id', 'player_position', 'player_in_possession_id', 'player_in_possession_name', 'player_in_possession_position_id', 'player_in_possession_position', 'team_id', 'team_shortname', 'x_start', 'y_start', 'channel_id_start', 'channel_start', 'third_id_start', 'third_start', 'penalty_area_start', 'x_end', 'y_end', 'channel_id_end', 'channel_end', 'third_id_end', 'third_end', 'penalty_area_end', 'associated_player_possession_event_id', 'associated_player_possession_frame_start', 'associated_player_possession_frame_end', 'associated_player_possession_end_type_id', 'associated_player_possession_end_type', 'associated_off_ball_run_event_id', 'associated_off_ball_run_subty

/tmp/ipykernel_3278/793838213.py:1: DtypeWarning: Columns (75,77,184,264) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv(f"{match_folder}/{match_id}_dynamic_events.csv")


,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,...,xloss_player_possession_end,xloss_player_possession_max,xshot_player_possession_start,xshot_player_possession_end,xshot_player_possession_max,is_player_possession_start_matched,is_player_possession_end_matched,is_previous_pass_matched,is_pass_reception_matched,fully_extrapolated
0,8_0,0,2017461,2512,2512,NaN,00:00.2,00:00.2,0,0,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,True,False
1,1_0,1,2017461,2523,2539,NaN,00:01.3,00:02.9,0,1,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN,NaN
2,8_1,2,2017461,2526,2540,NaN,00:01.6,00:03.0,0,1,...,NaN,NaN,NaN,NaN,NaN,True,True,True,True,False
3,7_0,3,2017461,2526,2528,NaN,00:01.6,00:01.8,0,1,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN,False
4,7_1,4,2017461,2526,2540,NaN,00:01.6,00:03.0,0,1,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN,False


## 10. Inspect Event Types

In [17]:
events["event_type"].value_counts().head(30)

,count
event_type,
passing_option,2068
player_possession,839
on_ball_engagement,821
off_ball_run,460


## 11. Inspect Event Subtypes

In [18]:
events["event_subtype"].value_counts().head(50)

,count
event_subtype,
pressure,232
other,208
recovery_press,202
run_ahead_of_the_ball,134
counter_press,91
pressing,88
support,82
coming_short,52
cross_receiver,46


In [19]:
events[[
    "event_type",
    "event_subtype",
    "associated_off_ball_run_subtype",
    "team_in_possession_phase_type",
    "team_out_of_possession_phase_type",
    "xthreat",
    "player_targeted_xthreat",
    "lead_to_shot",
    "lead_to_goal",
    "organised_defense",
    "defensive_structure"
]].head(20)

,event_type,event_subtype,associated_off_ball_run_subtype,team_in_possession_phase_type,team_out_of_possession_phase_type,xthreat,player_targeted_xthreat,lead_to_shot,lead_to_goal,organised_defense,defensive_structure
0,player_possession,NaN,NaN,create,medium_block,NaN,NaN,False,False,NaN,NaN
1,off_ball_run,run_ahead_of_the_ball,NaN,direct,defending_direct,0.0072,NaN,False,False,NaN,NaN
2,player_possession,NaN,NaN,direct,defending_direct,NaN,0.0072,False,False,True,442.0
3,passing_option,NaN,NaN,direct,defending_direct,0.0010,NaN,False,False,True,442.0
4,passing_option,NaN,coming_short,direct,defending_direct,0.0005,NaN,False,False,True,442.0
5,passing_option,NaN,run_ahead_of_the_ball,direct,defending_direct,0.0045,NaN,False,False,True,442.0
6,off_ball_run,coming_short,NaN,direct,defending_direct,0.0005,NaN,False,False,NaN,NaN
7,on_ball_engagement,pressure,NaN,direct,defending_direct,NaN,NaN,False,False,NaN,NaN
8,passing_option,NaN,run_ahead_of_the_ball,direct,defending_direct,0.0072,NaN,False,False,True,442.0
9,off_ball_run,run_ahead_of_the_ball,NaN,direct,defending_direct,0.0040,NaN,False,False,NaN,NaN


## 13. Create Clean Run Dataset

In [20]:
runs = events[events["event_type"] == "off_ball_run"]

runs[[
    "player_name",
    "event_subtype",
    "team_in_possession_phase_type",
    "team_out_of_possession_phase_type",
    "xthreat",
    "lead_to_shot",
    "lead_to_goal"
]].head(20)

,player_name,event_subtype,team_in_possession_phase_type,team_out_of_possession_phase_type,xthreat,lead_to_shot,lead_to_goal
1,Z. Machach,run_ahead_of_the_ball,direct,defending_direct,0.0072,False,False
6,J. Valadon,coming_short,direct,defending_direct,0.0005,False,False
9,N. Velupillay,run_ahead_of_the_ball,direct,defending_direct,0.0040,False,False
19,Z. Machach,support,finish,low_block,0.0519,False,False
20,D. Arzani,cross_receiver,finish,low_block,0.0403,False,False
21,N. Vergos,cross_receiver,finish,low_block,0.1566,False,False
73,Z. Machach,run_ahead_of_the_ball,chaotic,chaotic,0.0231,False,False
74,K. Bos,support,chaotic,chaotic,0.0077,False,False
80,N. Velupillay,run_ahead_of_the_ball,chaotic,chaotic,0.0206,False,False
88,M. Francois,run_ahead_of_the_ball,transition,defending_transition,0.0070,False,False


## 14. Inspect Run Frequency

In [21]:
runs["event_subtype"].value_counts()

,count
event_subtype,
run_ahead_of_the_ball,134
support,82
coming_short,52
cross_receiver,46
dropping_off,45
behind,28
pulling_wide,27
overlap,21
pulling_half_space,13


## 15. Compute Average Run Value

In [22]:
runs.groupby("event_subtype")["xthreat"].mean().sort_values(ascending=False)

,xthreat
event_subtype,
cross_receiver,0.113885
behind,0.040700
underlap,0.022375
support,0.021044
run_ahead_of_the_ball,0.014285
overlap,0.011595
pulling_half_space,0.006415
coming_short,0.002965
pulling_wide,0.002904


In [23]:
runs = events[events["event_type"] == "off_ball_run"]

runs_df = runs[[
    "player_name",
    "team_shortname",
    "event_subtype",
    "team_in_possession_phase_type",
    "team_out_of_possession_phase_type",
    "xthreat",
    "lead_to_shot",
    "lead_to_goal",
    "frame_start",
    "frame_end"
]].copy()

runs_df.head()

,player_name,team_shortname,event_subtype,team_in_possession_phase_type,team_out_of_possession_phase_type,xthreat,lead_to_shot,lead_to_goal,frame_start,frame_end
1,Z. Machach,Melbourne V FC,run_ahead_of_the_ball,direct,defending_direct,0.0072,False,False,2523,2539
6,J. Valadon,Melbourne V FC,coming_short,direct,defending_direct,0.0005,False,False,2529,2544
9,N. Velupillay,Melbourne V FC,run_ahead_of_the_ball,direct,defending_direct,0.0040,False,False,2541,2550
19,Z. Machach,Melbourne V FC,support,finish,low_block,0.0519,False,False,2629,2661
20,D. Arzani,Melbourne V FC,cross_receiver,finish,low_block,0.0403,False,False,2635,2658


In [24]:
runs_df.shape

(460, 10)

## 16. Save Run Dataset

In [25]:
runs_df.to_csv("runs_dataset.csv", index=False)